# EPA Water Quality: Merge Stations + Measurements

In [1]:
import numpy as np
import pandas as pd
import os
cleaned_path = r'../../data/tabular/water-quality/clean'

df_wq = pd.read_csv(
    f'{cleaned_path}/epa-wq-clean.csv',
    parse_dates=['ActivityStartDateTime']
)
df_station = pd.read_csv(f'{cleaned_path}/epa-stations-clean.csv')

print(f'Water quality rows: {len(df_wq):,}')
print(f'Stations rows:      {len(df_station):,}')

Water quality rows: 59,763
Stations rows:      621


In [2]:
# Step 1: Inspect join key overlap
wq_stations = set(df_wq['MonitoringLocationIdentifier'].dropna())
station_ids = set(df_station['MonitoringLocationIdentifier'].dropna())

print(f'Unique stations in WQ data:       {len(wq_stations)}')
print(f'Unique stations in stations file:  {len(station_ids)}')
print(f'Overlap:                           {len(wq_stations & station_ids)}')
print(f'WQ stations NOT in stations file:  {len(wq_stations - station_ids)}')

Unique stations in WQ data:       617
Unique stations in stations file:  621
Overlap:                           617
WQ stations NOT in stations file:  0


In [3]:
# Step 2: Merge water quality measurements with station metadata
# Left join keeps all WQ rows; station metadata added where available
df_merged = df_wq.merge(
    df_station,
    on='MonitoringLocationIdentifier',
    how='left'
)

print(f'Merged shape: {df_merged.shape}')
df_merged.head()

Merged shape: (59763, 19)


,ActivityIdentifier,MonitoringLocationIdentifier,MonitoringLocationName_x,ActivityLocation/LatitudeMeasure,ActivityLocation/LongitudeMeasure,CharacteristicName,ResultMeasureValue,ResultMeasure/MeasureUnitCode,ResultValueTypeName,ActivityStartDateTime,OrganizationIdentifier,MonitoringLocationName_y,MonitoringLocationTypeName,HUCEightDigitCode,LatitudeMeasure,LongitudeMeasure,StateCode,CountyCode,ProviderName
0,nwisia.01.02400517,USGS-05474000,NaN,NaN,NaN,"Temperature, water",0.0,deg C,Actual,2024-01-01 12:00:00,USGS-IA,"Skunk River at Augusta, IA",Stream,7080107,40.753650,-91.277089,19,57,NWIS
1,nwisia.01.02400551,USGS-05451210,NaN,NaN,NaN,Stream width measure,30.0,ft,Actual,2024-02-26 10:30:00,USGS-IA,"South Fork Iowa River NE of New Providence, IA",Stream,7080207,42.315306,-93.152194,19,83,NWIS
2,nwisia.01.02400551,USGS-05451210,NaN,NaN,NaN,"Temperature, water",4.7,deg C,Actual,2024-02-26 10:30:00,USGS-IA,"South Fork Iowa River NE of New Providence, IA",Stream,7080207,42.315306,-93.152194,19,83,NWIS
3,nwisia.01.02400551,USGS-05451210,NaN,NaN,NaN,"Temperature, air",8.9,deg C,Actual,2024-02-26 10:30:00,USGS-IA,"South Fork Iowa River NE of New Providence, IA",Stream,7080207,42.315306,-93.152194,19,83,NWIS
4,nwisia.01.02400551,USGS-05451210,NaN,NaN,NaN,Specific conductance,579.0,uS/cm @25C,Actual,2024-02-26 10:30:00,USGS-IA,"South Fork Iowa River NE of New Providence, IA",Stream,7080207,42.315306,-93.152194,19,83,NWIS


In [4]:
# Step 3: Resolve latitude/longitude columns
# WQ data has activity-level lat/lon; stations file has fixed station lat/lon
# Use station lat/lon as authoritative; fall back to activity lat/lon
wq_lat = 'ActivityLocation/LatitudeMeasure'
wq_lon = 'ActivityLocation/LongitudeMeasure'

if 'LatitudeMeasure' in df_merged.columns:
    df_merged['latitude'] = df_merged['LatitudeMeasure'].combine_first(
        pd.to_numeric(df_merged.get(wq_lat), errors='coerce')
    )
    df_merged['longitude'] = df_merged['LongitudeMeasure'].combine_first(
        pd.to_numeric(df_merged.get(wq_lon), errors='coerce')
    )
    df_merged.drop(columns=['LatitudeMeasure', 'LongitudeMeasure', wq_lat, wq_lon],
                   errors='ignore', inplace=True)
else:
    df_merged['latitude'] = pd.to_numeric(df_merged.get(wq_lat), errors='coerce')
    df_merged['longitude'] = pd.to_numeric(df_merged.get(wq_lon), errors='coerce')
    df_merged.drop(columns=[wq_lat, wq_lon], errors='ignore', inplace=True)

print(f'Missing latitude:  {df_merged["latitude"].isnull().sum()}')
print(f'Missing longitude: {df_merged["longitude"].isnull().sum()}')

Missing latitude:  0
Missing longitude: 0


In [5]:
# Step 4: Final inspection
print(f'Final shape: {df_merged.shape}')
print('\nColumns:', df_merged.columns.tolist())
print('\nMissing values:')
print(df_merged.isnull().sum())

Final shape: (59763, 17)

Columns: ['ActivityIdentifier', 'MonitoringLocationIdentifier', 'MonitoringLocationName_x', 'CharacteristicName', 'ResultMeasureValue', 'ResultMeasure/MeasureUnitCode', 'ResultValueTypeName', 'ActivityStartDateTime', 'OrganizationIdentifier', 'MonitoringLocationName_y', 'MonitoringLocationTypeName', 'HUCEightDigitCode', 'StateCode', 'CountyCode', 'ProviderName', 'latitude', 'longitude']

Missing values:
ActivityIdentifier                  0
MonitoringLocationIdentifier        0
MonitoringLocationName_x           80
CharacteristicName                  0
ResultMeasureValue                  0
ResultMeasure/MeasureUnitCode    6067
ResultValueTypeName                 0
ActivityStartDateTime             558
OrganizationIdentifier              0
MonitoringLocationName_y            0
MonitoringLocationTypeName          0
HUCEightDigitCode                   0
StateCode                           0
CountyCode                          0
ProviderName                       

In [6]:
df_merged["MonitoringLocationName_x"].isnull().sum() * 100 / len(df_merged)

np.float64(0.13386208858323712)

In [7]:
df_merged.drop(columns=['MonitoringLocationName_x'], inplace=True)
df_merged.rename(columns={'MonitoringLocationName_y': 'MonitoringLocationName'}, inplace=True)

In [8]:
df_merged.head()

,ActivityIdentifier,MonitoringLocationIdentifier,CharacteristicName,ResultMeasureValue,ResultMeasure/MeasureUnitCode,ResultValueTypeName,ActivityStartDateTime,OrganizationIdentifier,MonitoringLocationName,MonitoringLocationTypeName,HUCEightDigitCode,StateCode,CountyCode,ProviderName,latitude,longitude
0,nwisia.01.02400517,USGS-05474000,"Temperature, water",0.0,deg C,Actual,2024-01-01 12:00:00,USGS-IA,"Skunk River at Augusta, IA",Stream,7080107,19,57,NWIS,40.753650,-91.277089
1,nwisia.01.02400551,USGS-05451210,Stream width measure,30.0,ft,Actual,2024-02-26 10:30:00,USGS-IA,"South Fork Iowa River NE of New Providence, IA",Stream,7080207,19,83,NWIS,42.315306,-93.152194
2,nwisia.01.02400551,USGS-05451210,"Temperature, water",4.7,deg C,Actual,2024-02-26 10:30:00,USGS-IA,"South Fork Iowa River NE of New Providence, IA",Stream,7080207,19,83,NWIS,42.315306,-93.152194
3,nwisia.01.02400551,USGS-05451210,"Temperature, air",8.9,deg C,Actual,2024-02-26 10:30:00,USGS-IA,"South Fork Iowa River NE of New Providence, IA",Stream,7080207,19,83,NWIS,42.315306,-93.152194
4,nwisia.01.02400551,USGS-05451210,Specific conductance,579.0,uS/cm @25C,Actual,2024-02-26 10:30:00,USGS-IA,"South Fork Iowa River NE of New Providence, IA",Stream,7080207,19,83,NWIS,42.315306,-93.152194


In [9]:
# Step 5: Save merged data
out_path = '../../data/tabular/merged'
os.makedirs(out_path, exist_ok=True)
df_merged.to_csv(f'{out_path}/epa-merged.csv', index=False)
print(f'Saved {len(df_merged):,} rows → {out_path}/epa-merged.csv')

Saved 59,763 rows → ../../data/tabular/merged/epa-merged.csv
